In [2]:
import os
import numpy as np
from openai import OpenAI
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import chromadb

current_directory = Path.cwd()
path = f"{current_directory}\chroma_db"

client = chromadb.PersistentClient(path=path)

collection = client.get_collection('bbc_news')

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

hg_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

d:\AI Projects\RAG Practice\BBC_News_Extraction\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2294.87it/s]


In [5]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1156.31it/s]


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_context(query):
    query_embedding = hg_embeddings.embed_query(query)

    magnitude = np.linalg.norm(query_embedding)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        include=["documents", "embeddings", "distances"]
    )
    print(results["distances"][0])

    # result_embeddings = results["embeddings"][0]
    # similarities = cosine_similarity(
    #     [query_embedding],
    #     result_embeddings
    # )[0]

    # print(similarities)

    print("\nFetched relevant documents:")
    print("---------------------------")
    documents = results["documents"][0]
    for doc in documents:
        print(doc,"\n")
        
    print("=================================================================================================================")

    pairs = [
        (query, document)
        for document in documents
    ]

    scores = reranker.predict(pairs)
    print(scores)

    scores_map = list(zip(scores, documents))
    scores_map.sort(reverse=True)

    ranked_docs = [doc for score, doc in scores_map[:3]]
    return ranked_docs

In [7]:
def call_llm_with_rag(query):
    chunks = retrieve_context(query)
    context = "\n\n".join(chunks)

    print("\nReranked useful documents:")
    print("--------------------------")
    print(context)

    API_KEY = os.getenv('REQUESTY_API_KEY')

    client = OpenAI(
        base_url="https://router.requesty.ai/v1",
        api_key=API_KEY
    )
    
    system_prompt = f""""You are an expert summarizer and analyst.
Answer the user's question using only the information provided in the context.
Along with the response, provide supporting evidence specifying the documents that contain that information.
Do not introduce facts that are not supported by the context.
If the context is not sufficient to answer the question, just tell that without giving further explanation about other chunks.
"""

    user_prompt = f"""Context: {context}\n
Question: {query}
"""
 
    response = client.chat.completions.create(
        # model="nvidia/nemotron-3-ultra-550b-a55b",
        model="google/gemma-4-31b-it",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    answer = response.choices[0].message.content
    return answer

In [8]:
query = input()
answer = call_llm_with_rag(query)

[0.357535719871521, 0.41373443603515625, 0.4548739194869995, 0.4893321990966797, 0.4953291416168213]

Fetched relevant documents:
---------------------------
warnings about junk mail deluge the amount of spam circulating online could be about to undergo a massive increase say experts antispam group spamhaus is warning about a novel virus which hides the origins of junk mail the program makes spam look like it is being sent by legitimate mail servers making it hard to spot and filter out spamhaus said that if the problem went unchecked real email messages could get drowned by the sheer amount of junk being sent before now many spammers have recruited home pcs to act as anonymous email relays in an attempt to hide the origins of their junk mail the pcs are recruited using viruses and worms that compromise machines via known vulnerabilities or by tricking people into opening an attachment infected with the malicious program once compromised the machines start to pump out junk mail on beha

In [9]:
print(answer)

if answer:
    content = f"""Qn: {query}
Ans:
{answer}

---------------------------------------------------------------------------------------------\n
"""

    with open('llm_responses.txt', 'a', encoding='utf-8') as f:
        f.write(content)

Based on the provided text, the following steps are taken to reduce spam emails:

*   **Consumer-level protections:** Users are encouraged to use a combination of sound judgement, spyware detection software, and spam filters.
*   **Industry and media efforts:** Both the media and industry have worked to raise awareness regarding illegitimate emails to reduce nuisance and financial damage from spoof websites and phishing attacks.
*   **Blacklisting and blocking:** 
    *   Spamhaus blocks junk messages by collecting and circulating blacklists of net addresses known to harbor infected machines.
    *   Organizations such as Spamcop generate real-time blacklists of sites that sell spam goods.
*   **Technical filtering:** Symantec's subsidiary, Brightmail, utilizes filtering techniques that do not rely on net addresses, such as filtering out email messages that contain web links.
*   **Increasing spammers' costs:** Lycos developed a screensaver that endlessly requests data from sites selli